# 02. Agent Execution Loop

This notebook builds a production-grade, bounded execution loop. We demonstrate:
- Explicit state modeling with observable action rationales (not hidden chain-of-thought)
- Strict error classification: transient transport timeouts vs correctable schema validation vs fatal policy blocks
- Per-tool retry budgets (avoiding cross-tool budget starvation)
- No-progress loop detection based on repeated action-observation fingerprints
- Dynamic hypothesis replanning when new evidence contradicts assumptions
- Durable state machine mapping with LangGraph

In [1]:
import json
import time
import os
from enum import Enum
from typing import Dict, Any, List, Optional, Literal, Set, Callable
from pydantic import BaseModel, Field, ValidationError

# Central model configuration
MODEL_NAME = 'gpt-4o-mini'
print(f'Environment initialized. Configured model: {MODEL_NAME}')

Environment initialized. Configured model: gpt-4o-mini


## Part 1: The Pitfall of the Naive Agent Loop

A naive `while` loop that retries indefinitely without budget caps, error classification, or loop detection will run runaway iterations on transient or repeated errors.

In [2]:
# Demonstration of naive unguided looping
def naive_decide(state_dict: dict) -> str:
    if 'error' in str(state_dict.get('last_obs', '')).lower():
        return 'retry_same_action'
    return 'final_answer'

state = {'last_obs': 'API Error: HTTP 504 Timeout'}
steps = 0
print('Starting naive loop...')
while state.get('last_obs') != 'success':
    action = naive_decide(state)
    state['last_obs'] = 'API Error: HTTP 504 Timeout' # Simulating recurring failure
    steps += 1
    if steps >= 4:
        print(f'DANGER: Runaway loop reached {steps} steps without progress. Forcing exit.')
        break

Starting naive loop...
DANGER: Runaway loop reached 4 steps without progress. Forcing exit.


## Part 2: Typed State, Tool Calls & Observable Decisions

In production, every transition must be inspectable. `AgentDecision` captures an observable `decision_summary` rather than unverified hidden chain-of-thought. Retry budgets are tracked **per-tool** (`retries_by_tool`) so that a transient timeout on one service does not consume the budget for subsequent tools.

In [3]:
class ToolCall(BaseModel):
    tool: Literal['get_service_health', 'query_checkout_logs', 'search_incidents', 'get_runbook', 'get_recent_deployments']
    arguments: Dict[str, Any] = Field(default_factory=dict)

class AgentDecision(BaseModel):
    decision_summary: str = Field(..., description='Observable summary of the action rationale')
    tool_call: Optional[ToolCall] = None
    final_answer: Optional[str] = None
    replan: bool = False

class AgentState(BaseModel):
    goal: str
    scenario_type: str = 'happy_path'
    evidence: List[str] = Field(default_factory=list)
    steps: int = 0
    tool_calls: int = 0
    retries_by_tool: Dict[str, int] = Field(default_factory=dict)
    replans: int = 0
    seen_action_fingerprints: Set[str] = Field(default_factory=set)
    terminal_reason: Optional[str] = None
    current_hypothesis: Optional[str] = None
    replan_reason: Optional[str] = None
    history: List[Dict[str, Any]] = Field(default_factory=list)

print('Domain State Models Initialized.')

Domain State Models Initialized.


## Part 3: Explicit Tool Registry & Typed Handlers

Each tool defines a strict Pydantic argument model and an explicit `max_retries` policy for transient errors.

In [4]:
# Pydantic Tool Input Schemas
class RegionQuery(BaseModel):
    region: str = Field(..., description='AWS/GCP region code, e.g. EU, US')

class ServiceQuery(BaseModel):
    service: str = Field(..., description='Name of microservice')

class TopicQuery(BaseModel):
    topic: str = Field(..., description='Error topic or symptom key')

# Tool Handlers simulating realistic environment responses
def get_service_health(args: RegionQuery) -> str:
    if args.region == 'FAIL_TIMEOUT':
        raise TimeoutError('HTTP 504 Gateway Timeout connecting to health check endpoint')
    if args.region == 'EU':
        return 'Status: Degraded (Checkout DB High Latency on route /v1/checkout)'
    return 'Status: Healthy (All systems operational)'

def query_checkout_logs(args: RegionQuery) -> str:
    if args.region == 'DENY_ACCESS':
        raise PermissionError('Permission Denied: Caller lacks sec-read permission on confidential log stream')
    if args.region == 'EU':
        return '[EU-Logs] 45 upstream timeout exceptions matching Stripe payment gateway.'
    return 'No matching error logs found.'

def search_incidents(args: ServiceQuery) -> str:
    return f'No active incident record found for service {args.service}.'

def get_runbook(args: TopicQuery) -> str:
    if 'timeout' in args.topic.lower():
        return 'Runbook: Check recent service deployments. If recent deployment found within 30 min, verify changelog.'
    return 'Runbook: Standard diagnostic flow.'

def get_recent_deployments(args: ServiceQuery) -> str:
    if args.service == 'checkout':
        return 'Deployment v1.14 shipped 20 minutes ago to checkout-api by deploy-bot.'
    return 'No recent deployments.'

# Tool Registry Definition
TOOL_REGISTRY = {
    'get_service_health': {'func': get_service_health, 'schema': RegionQuery, 'read_only': True, 'max_retries': 2},
    'query_checkout_logs': {'func': query_checkout_logs, 'schema': RegionQuery, 'read_only': True, 'max_retries': 2},
    'search_incidents': {'func': search_incidents, 'schema': ServiceQuery, 'read_only': True, 'max_retries': 2},
    'get_runbook': {'func': get_runbook, 'schema': TopicQuery, 'read_only': True, 'max_retries': 2},
    'get_recent_deployments': {'func': get_recent_deployments, 'schema': ServiceQuery, 'read_only': True, 'max_retries': 2}
}
print('Tool Registry initialized with 5 typed tools.')

Tool Registry initialized with 5 typed tools.


## Part 4: Dispatcher with Error Classification & Bounded Loop

### Strict Error Taxonomy
We categorize execution failures into 4 distinct classes:
1. **`INVALID_TOOL_ARGS`** (`ValidationError`): Model generated improper arguments. The error is returned to the model for bounded correction.
2. **`POLICY_BLOCK`** (`PermissionError`): Explicit authorization failure. Terminate immediately; never blindly retry.
3. **`TRANSIENT_TIMEOUT`** (`TimeoutError`): Network/transport glitch. Retry up to the tool's individual `max_retries` budget.
4. **`TOOL_FAILURE`** (`Exception`): Unhandled execution error.

### No-Progress Loop Detection
Detects when the agent produces the **exact same action fingerprint** and receives the **exact same observation** repeatedly without updating hypothesis or evidence.

In [5]:
class DispatchStatus(str, Enum):
    SUCCESS = 'SUCCESS'
    INVALID_TOOL_ARGS = 'INVALID_TOOL_ARGS'
    POLICY_BLOCK = 'POLICY_BLOCK'
    TRANSIENT_TIMEOUT = 'TRANSIENT_TIMEOUT'
    TOOL_FAILURE = 'TOOL_FAILURE'

class DispatchResult(BaseModel):
    status: DispatchStatus
    output: str

def dispatch_tool(tool_name: str, args: dict) -> DispatchResult:
    if tool_name not in TOOL_REGISTRY:
        return DispatchResult(status=DispatchStatus.POLICY_BLOCK, output=f"Tool '{tool_name}' is not in allowed registry.")
    
    entry = TOOL_REGISTRY[tool_name]
    try:
        validated_args = entry['schema'](**args)
        raw_result = entry['func'](validated_args)
        return DispatchResult(status=DispatchStatus.SUCCESS, output=str(raw_result))
    except ValidationError as e:
        # Schema errors are model argument issues, NOT policy blocks
        return DispatchResult(status=DispatchStatus.INVALID_TOOL_ARGS, output=f"Schema ValidationError: {e.errors()[0]['msg']}")
    except PermissionError as e:
        # Authorization/security denial is a fatal policy block
        return DispatchResult(status=DispatchStatus.POLICY_BLOCK, output=f"Policy Authorization Error: {str(e)}")
    except TimeoutError as e:
        # Transient transport timeout
        return DispatchResult(status=DispatchStatus.TRANSIENT_TIMEOUT, output=f"Transient Timeout: {str(e)}")
    except Exception as e:
        return DispatchResult(status=DispatchStatus.TOOL_FAILURE, output=f"Unhandled Tool Execution Error: {str(e)}")

def run_bounded_loop(state: AgentState, decision_model, max_steps: int = 8) -> AgentState:
    print(f"\n[Runtime Started] Goal: {state.goal}")
    last_action_fingerprint = None
    last_observation = None
    no_progress_count = 0

    while state.steps < max_steps:
        state.steps += 1
        decision: AgentDecision = decision_model(state)
        state.history.append({'role': 'model', 'decision': decision})
        print(f"\nStep {state.steps} | Action rationale: {decision.decision_summary}")
        if state.current_hypothesis:
            print(f"Current Hypothesis: {state.current_hypothesis}")

        if decision.final_answer:
            print(f"[Terminal] Final Answer: {decision.final_answer}")
            state.terminal_reason = 'SUCCESS'
            break

        if decision.tool_call:
            t_name = decision.tool_call.tool
            args = decision.tool_call.arguments
            action_fingerprint = f"{t_name}:{json.dumps(args, sort_keys=True)}"

            # Dispatch with typed error classification
            dispatch_res = dispatch_tool(t_name, args)
            print(f"[Dispatch Result] ({dispatch_res.status.value}): {dispatch_res.output}")

            # 1. Fatal Authorization / Policy Denial
            if dispatch_res.status == DispatchStatus.POLICY_BLOCK:
                print("FATAL POLICY BLOCK: Security violation or unauthorized access. Terminating immediately.")
                state.terminal_reason = 'POLICY_BLOCK'
                break

            # 2. Transient Timeout with Per-Tool Retry Budget
            if dispatch_res.status == DispatchStatus.TRANSIENT_TIMEOUT:
                current_retries = state.retries_by_tool.get(t_name, 0)
                max_allowed = TOOL_REGISTRY[t_name]['max_retries']
                if current_retries < max_allowed:
                    state.retries_by_tool[t_name] = current_retries + 1
                    print(f"TRANSIENT RETRY: Retrying {t_name} (Attempt {current_retries + 1}/{max_allowed})...")
                    continue
                else:
                    print(f"RETRY BUDGET EXHAUSTED: Tool {t_name} exceeded max retries ({max_allowed}).")
                    state.terminal_reason = 'TOOL_FAILURE'
                    break

            # 3. Schema / Argument Validation Error (Feed back to model for bounded correction)
            if dispatch_res.status == DispatchStatus.INVALID_TOOL_ARGS:
                state.history.append({'role': 'environment', 'tool': t_name, 'error': dispatch_res.output})
                # Check if model repeatedly makes the same schema error
                if action_fingerprint == last_action_fingerprint:
                    print("CORRECTION FAILED: Model repeated identical invalid arguments. Terminating.")
                    state.terminal_reason = 'INVALID_TOOL_ARGS'
                    break
                last_action_fingerprint = action_fingerprint
                continue

            # 4. Successful Observation
            state.tool_calls += 1
            state.seen_action_fingerprints.add(action_fingerprint)
            state.evidence.append(dispatch_res.output)
            state.history.append({'role': 'environment', 'tool': t_name, 'observation': dispatch_res.output})

            # 5. No-Progress Detection (Identical Action + Identical Observation without state mutation)
            if action_fingerprint == last_action_fingerprint and dispatch_res.output == last_observation:
                no_progress_count += 1
            else:
                no_progress_count = 0
            last_action_fingerprint = action_fingerprint
            last_observation = dispatch_res.output

            if no_progress_count >= 2:
                print("NO PROGRESS DETECTED: Agent executed repeated action with identical observation. Escalating.")
                state.terminal_reason = 'NO_PROGRESS'
                break

    if not state.terminal_reason:
        state.terminal_reason = 'STEP_BUDGET_EXHAUSTED'
    print(f"[Runtime Finished] Terminal reason: {state.terminal_reason}")
    return state

## Part 5: Deterministic Decision Model

We define a multi-scenario deterministic decision model to rigorously verify the execution invariants.

In [6]:
def mock_decision_model(state: AgentState) -> AgentDecision:
    scen = state.scenario_type

    if scen == 'happy_path':
        if state.tool_calls == 0:
            return AgentDecision(decision_summary='Check regional service health for EU.', tool_call=ToolCall(tool='get_service_health', arguments={'region': 'EU'}))
        if state.tool_calls == 1:
            return AgentDecision(decision_summary='Inspect error logs for EU checkout.', tool_call=ToolCall(tool='query_checkout_logs', arguments={'region': 'EU'}))
        return AgentDecision(decision_summary='Synthesize diagnostic findings.', final_answer='EU Checkout degraded due to payment gateway timeouts.')

    if scen == 'replan':
        if state.tool_calls == 0:
            state.current_hypothesis = 'Network routing outage'
            return AgentDecision(decision_summary='Initial hypothesis: Network issue. Check logs.', tool_call=ToolCall(tool='query_checkout_logs', arguments={'region': 'EU'}))
        if state.tool_calls == 1:
            return AgentDecision(decision_summary='Timeouts found in logs. Check timeout runbook.', tool_call=ToolCall(tool='get_runbook', arguments={'topic': 'timeout'}))
        if state.tool_calls == 2:
            return AgentDecision(decision_summary='Runbook suggests checking recent deployments.', tool_call=ToolCall(tool='get_recent_deployments', arguments={'service': 'checkout'}))
        if state.tool_calls == 3:
            # New evidence invalidates initial network outage hypothesis
            state.current_hypothesis = 'Bad deployment v1.14 shipped 20m ago'
            state.replan_reason = 'Deployment v1.14 evidence contradicts network outage hypothesis.'
            state.replans += 1
            print(f"\n>>> DYNAMIC REPLAN TRIGGERED: {state.replan_reason} <<<")
            return AgentDecision(decision_summary='Replan based on deployment evidence.', final_answer='Root cause: Deployment v1.14 caused checkout regression. Rollback recommended.')

    if scen == 'malformed_args_recovery':
        # Turn 0 proposes bad args ('reg' instead of 'region') -> triggers INVALID_TOOL_ARGS
        # Turn 1 receives schema feedback and corrects args -> succeeds
        # Turn 2 concludes with final answer
        has_schema_err = any('Schema ValidationError' in str(h.get('error', '')) for h in state.history)
        if not has_schema_err:
            return AgentDecision(decision_summary='Propose query with malformed argument key.', tool_call=ToolCall(tool='get_service_health', arguments={'reg': 'EU'}))
        elif state.tool_calls == 0:
            return AgentDecision(decision_summary='Correct argument key to region after schema validation feedback.', tool_call=ToolCall(tool='get_service_health', arguments={'region': 'EU'}))
        else:
            return AgentDecision(decision_summary='Conclude after successful corrected call.', final_answer='Investigation complete after correcting tool arguments.')

    if scen == 'transient_timeout_exhausted':
        # Simulates repeated timeout until per-tool retry budget is exhausted
        return AgentDecision(decision_summary='Query endpoint that produces repeated 504 timeouts.', tool_call=ToolCall(tool='get_service_health', arguments={'region': 'FAIL_TIMEOUT'}))

    if scen == 'permission_denied':
        # Simulates fatal policy authorization block
        return AgentDecision(decision_summary='Attempt query on restricted log partition.', tool_call=ToolCall(tool='query_checkout_logs', arguments={'region': 'DENY_ACCESS'}))

    if scen == 'repeated_action_loop':
        # Proposes identical action repeatedly without changing state
        return AgentDecision(decision_summary='Repeat identical log query without adapting.', tool_call=ToolCall(tool='query_checkout_logs', arguments={'region': 'EU'}))

    if scen == 'insufficient_evidence':
        if state.tool_calls == 0:
            return AgentDecision(decision_summary='Query US region.', tool_call=ToolCall(tool='get_service_health', arguments={'region': 'US'}))
        return AgentDecision(decision_summary='Abstain when evidence is inconclusive.', final_answer='Insufficient evidence to declare an incident in US region.')

    return AgentDecision(decision_summary='Default fallback', final_answer='Abort')

## Part 6: Multi-Scenario Evaluation Suite & Invariant Verification

We test the loop across 7 distinct operational scenarios and assert expected terminal states.

In [7]:
scenarios = [
    ('happy_path', 'happy_path', 'SUCCESS'),
    ('transient_timeout_exhausted', 'transient_timeout_exhausted', 'TOOL_FAILURE'),
    ('permission_denied', 'permission_denied', 'POLICY_BLOCK'),
    ('malformed_args_recovery', 'malformed_args_recovery', 'SUCCESS'),
    ('repeated_action_loop', 'repeated_action_loop', 'NO_PROGRESS'),
    ('changed_deployment_replan', 'replan', 'SUCCESS'),
    ('insufficient_evidence', 'insufficient_evidence', 'SUCCESS')
]

results = []
print('=== RUNNING COURSE 02 EVALUATION HARNESS ===')
for name, stype, expected_reason in scenarios:
    st = AgentState(goal=f'Scenario evaluation: {name}', scenario_type=stype)
    final_st = run_bounded_loop(st, mock_decision_model, max_steps=8)
    total_retries = sum(final_st.retries_by_tool.values())
    passed = (final_st.terminal_reason == expected_reason)
    results.append({
        'scenario': name,
        'passed': passed,
        'expected': expected_reason,
        'actual': final_st.terminal_reason,
        'steps': final_st.steps,
        'tool_calls': final_st.tool_calls,
        'retries': total_retries,
        'replans': final_st.replans
    })

print('\n=== EVALUATION RESULTS ===')
import pandas as pd
df = pd.DataFrame(results)
print(df.to_string(index=False))

# Assert all scenarios met exact expected terminal reasons
for r in results:
    assert r['passed'], f"Scenario {r['scenario']} expected {r['expected']} but got {r['actual']}"
print('\nAll 7 evaluation scenarios passed with verified terminal reasons!')

=== RUNNING COURSE 02 EVALUATION HARNESS ===

[Runtime Started] Goal: Scenario evaluation: happy_path

Step 1 | Action rationale: Check regional service health for EU.
[Dispatch Result] (SUCCESS): Status: Degraded (Checkout DB High Latency on route /v1/checkout)

Step 2 | Action rationale: Inspect error logs for EU checkout.
[Dispatch Result] (SUCCESS): [EU-Logs] 45 upstream timeout exceptions matching Stripe payment gateway.

Step 3 | Action rationale: Synthesize diagnostic findings.
[Terminal] Final Answer: EU Checkout degraded due to payment gateway timeouts.
[Runtime Finished] Terminal reason: SUCCESS

[Runtime Started] Goal: Scenario evaluation: transient_timeout_exhausted

Step 1 | Action rationale: Query endpoint that produces repeated 504 timeouts.
[Dispatch Result] (TRANSIENT_TIMEOUT): Transient Timeout: HTTP 504 Gateway Timeout connecting to health check endpoint
TRANSIENT RETRY: Retrying get_service_health (Attempt 1/2)...

Step 2 | Action rationale: Query endpoint that prod

                   scenario  passed     expected       actual  steps  tool_calls  retries  replans
                 happy_path    True      SUCCESS      SUCCESS      3           2        0        0
transient_timeout_exhausted    True TOOL_FAILURE TOOL_FAILURE      3           0        2        0
          permission_denied    True POLICY_BLOCK POLICY_BLOCK      1           0        0        0
    malformed_args_recovery    True      SUCCESS      SUCCESS      3           1        0        0
       repeated_action_loop    True  NO_PROGRESS  NO_PROGRESS      3           3        0        0
  changed_deployment_replan    True      SUCCESS      SUCCESS      4           3        0        1
      insufficient_evidence    True      SUCCESS      SUCCESS      2           1        0        0

All 7 evaluation scenarios passed with verified terminal reasons!


## Part 7: Reflection and Plan-and-Execute

Agents can evaluate intermediate steps using structured validation rubrics before emitting consequential actions.

In [8]:
print('--- Reflection & Grounding Rubric ---')
def evaluate_reflection_rubric(draft: str, evidence: list[str]) -> bool:
    print(f"Evaluating candidate recommendation: '{draft}'")
    if 'rollback' in draft.lower() and not any('Deployment' in e for e in evidence):
        print('-> Rubric Violation: Proposing rollback without evidence of recent deployment.')
        return False
    if 'restart' in draft.lower():
        print('-> Rubric Violation: Proposing restart without authorization.')
        return False
    print('-> Rubric Passed: Recommendation is grounded in collected evidence.')
    return True

assert not evaluate_reflection_rubric('We should restart db and rollback.', evidence=[])
assert evaluate_reflection_rubric('Recommend rollback of v1.14.', evidence=['Deployment v1.14 shipped 20 minutes ago'])
print('\nReflection rubric assertions verified.')

--- Reflection & Grounding Rubric ---
Evaluating candidate recommendation: 'We should restart db and rollback.'
-> Rubric Violation: Proposing rollback without evidence of recent deployment.
Evaluating candidate recommendation: 'Recommend rollback of v1.14.'
-> Rubric Passed: Recommendation is grounded in collected evidence.

Reflection rubric assertions verified.


## Part 8: Event-Driven Resumption (Durable Execution)

Durable execution requires that incoming events are processed idempotently using deduplication keys.

In [9]:
class WorkflowState:
    def __init__(self, workflow_id: str):
        self.workflow_id = workflow_id
        self.processed_events: Set[str] = set()
        self.resumed_state: Dict[str, Any] = {}

workflow_db: Dict[str, WorkflowState] = {}

def process_event(workflow_id: str, event_id: str, payload: str):
    if workflow_id not in workflow_db:
        workflow_db[workflow_id] = WorkflowState(workflow_id)
    wf = workflow_db[workflow_id]
    if event_id in wf.processed_events:
        print(f"Idempotency Guard: Event {event_id} already processed for {workflow_id}. Skipping duplicate delivery.")
        return
    print(f"Processing event {event_id} for {workflow_id}: {payload}")
    wf.processed_events.add(event_id)
    wf.resumed_state['last_event'] = payload

process_event('WF-100', 'EVT-1', 'Alert: Checkout 504 spike')
process_event('WF-100', 'EVT-1', 'Alert: Checkout 504 spike (Duplicate delivery)')

Processing event EVT-1 for WF-100: Alert: Checkout 504 spike
Idempotency Guard: Event EVT-1 already processed for WF-100. Skipping duplicate delivery.


## Part 9: State Machine Graph Orchestration (LangGraph)

Instead of manual `while` loops, production systems model transitions as explicit nodes and conditional edges.

In [10]:
try:
    from langgraph.graph import StateGraph, END
    from typing import TypedDict

    class GraphState(TypedDict):
        goal: str
        tool_calls: int
        terminal_reason: str

    def decide_node(state: GraphState):
        print(f"[LangGraph Node: Decide] Current tool_calls={state['tool_calls']}")
        return {'tool_calls': state['tool_calls'] + 1}

    def tool_node(state: GraphState):
        print('[LangGraph Node: Tool] Executing diagnostic tool...')
        return {}

    def evaluate_evidence_node(state: GraphState):
        print('[LangGraph Node: Evaluate] Inspecting evidence sufficiency...')
        if state['tool_calls'] >= 2:
            return {'terminal_reason': 'SUCCESS'}
        return {}

    def should_continue(state: GraphState) -> str:
        if state.get('terminal_reason') == 'SUCCESS':
            return 'recommend'
        return 'tool'

    def recommend_node(state: GraphState):
        print('[LangGraph Node: Recommend] Emitting grounded diagnosis.')
        return {}

    workflow = StateGraph(GraphState)
    workflow.add_node('decide', decide_node)
    workflow.add_node('tool', tool_node)
    workflow.add_node('evaluate', evaluate_evidence_node)
    workflow.add_node('recommend', recommend_node)

    workflow.set_entry_point('decide')
    workflow.add_edge('decide', 'evaluate')
    workflow.add_conditional_edges('evaluate', should_continue, {'tool': 'tool', 'recommend': 'recommend'})
    workflow.add_edge('tool', 'decide')
    workflow.add_edge('recommend', END)

    app = workflow.compile()
    print('\n--- LangGraph Compiled: Executing State Graph ---')
    graph_out = app.invoke({'goal': 'Diagnose EU checkout', 'tool_calls': 0, 'terminal_reason': ''})
    print('LangGraph execution completed:', graph_out)
except ImportError:
    print('LangGraph not installed. See architectural pattern above.')


--- LangGraph Compiled: Executing State Graph ---
[LangGraph Node: Decide] Current tool_calls=0
[LangGraph Node: Evaluate] Inspecting evidence sufficiency...
[LangGraph Node: Tool] Executing diagnostic tool...
[LangGraph Node: Decide] Current tool_calls=1
[LangGraph Node: Evaluate] Inspecting evidence sufficiency...
[LangGraph Node: Recommend] Emitting grounded diagnosis.
LangGraph execution completed: {'goal': 'Diagnose EU checkout', 'tool_calls': 2, 'terminal_reason': 'SUCCESS'}


## Part 10: Optional Live Model Execution (OpenAI API)

*(Optional)* When `OPENAI_API_KEY` is present, we execute the bounded loop with `gpt-4o-mini`, mapping model output directly into our internal `AgentDecision` schema.

In [11]:
api_key = os.getenv('OPENAI_API_KEY')
if not api_key:
    print('No OPENAI_API_KEY detected in environment. Skipping live OpenAI API call.')
else:
    from openai import OpenAI
    client = OpenAI(api_key=api_key)

    openai_tools = [
        {
            'type': 'function',
            'function': {
                'name': 'get_service_health',
                'description': 'Check regional service health',
                'parameters': {'type': 'object', 'properties': {'region': {'type': 'string'}}, 'required': ['region']}
            }
        },
        {
            'type': 'function',
            'function': {
                'name': 'query_checkout_logs',
                'description': 'Search checkout error logs',
                'parameters': {'type': 'object', 'properties': {'region': {'type': 'string'}}, 'required': ['region']}
            }
        }
    ]

    def openai_decision(state: AgentState) -> AgentDecision:
        messages = [
            {'role': 'system', 'content': 'You are a diagnostic assistant. Use tools to investigate incidents.'},
            {'role': 'user', 'content': state.goal}
        ]
        for h in state.history:
            if h['role'] == 'model' and h['decision'].tool_call:
                messages.append({
                    'role': 'assistant',
                    'content': h['decision'].decision_summary,
                    'tool_calls': [{
                        'id': 'call_01',
                        'type': 'function',
                        'function': {
                            'name': h['decision'].tool_call.tool,
                            'arguments': json.dumps(h['decision'].tool_call.arguments)
                        }
                    }]
                })
            elif h['role'] == 'environment':
                messages.append({
                    'role': 'tool',
                    'tool_call_id': 'call_01',
                    'name': h.get('tool', 'tool'),
                    'content': h.get('observation') or h.get('error', '')
                })

        response = client.chat.completions.create(model=MODEL_NAME, messages=messages, tools=openai_tools)
        msg = response.choices[0].message

        if msg.tool_calls:
            tc = msg.tool_calls[0].function
            try:
                args = json.loads(tc.arguments)
            except json.JSONDecodeError:
                args = {}
            return AgentDecision(decision_summary=f'Propose {tc.name} query.', tool_call=ToolCall(tool=tc.name, arguments=args))
        else:
            return AgentDecision(decision_summary='Emit final diagnosis.', final_answer=msg.content or '')

    print(f'\n--- Running Live Model in Bounded Loop ({MODEL_NAME}) ---')
    live_st = run_bounded_loop(AgentState(goal='Diagnose EU checkout health'), openai_decision, max_steps=5)
    print('Live run finished with reason:', live_st.terminal_reason)

No OPENAI_API_KEY detected in environment. Skipping live OpenAI API call.
